# Random Edge Addition testing

### TUDataset

In [ ]:
from torch_geometric import datasets
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCN
from rewiring.TrainValTestSplitter import TrainValTestSplitter
from rewiring.transform.AddRandomEdges import AddRandomEdges
from rewiring.pipeline.GraphClassificationPipeline import GraphClassificationPipeline

import torch
import numpy as np
import logging

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = AddRandomEdges(min_density=0.4)

batch = datasets.TUDataset(root=f'./../res/datasets/TUDataset', name='PROTEINS', transform=transform, use_node_attr=True, use_edge_attr=True)
# batch = datasets.TUDataset(root=f'./../res/datasets/TUDataset', name='MUTAG', transform=transform, use_node_attr=True, use_edge_attr=True)
# batch = datasets.TUDataset(root=f'./../res/datasets/TUDataset', name='ENZYMES', transform=transform, use_node_attr=True, use_edge_attr=True)
# batch = datasets.TUDataset(root=f'./../res/datasets/TUDataset', name='COLLAB', transform=transform, use_node_attr=True, use_edge_attr=True)

Processing...
Done!


In [ ]:
print(f"Number of graphs: {len(batch)}")
print(batch.name)
val_ratio = 0.2
train_dataset, val_dataset, test_dataset = TrainValTestSplitter.graph_class_split(batch, val_size=val_ratio, test_size=1)
val_count = len(val_dataset)

batch_size = 64

train_dloader: DataLoader = DataLoader(train_dataset, 
                                       shuffle=True, 
                                       batch_size=batch_size)

val_dloader: DataLoader = DataLoader(val_dataset, 
                                      shuffle=False, 
                                      batch_size=batch_size)

test_loader: DataLoader = DataLoader(test_dataset, 
                                      shuffle=False, 
                                      batch_size=batch_size)

print("Creating model...")
model = GCN(out_channels=batch.num_classes, 
            in_channels=batch.num_features, 
            hidden_channels=16,
            num_layers=4)
print("Model created!")

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.NLLLoss()

pipeline = GraphClassificationPipeline(
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    model=model,
    criterion=criterion,
    optimizer=optimizer
)

_, _, _ = pipeline.train_validate(
    train_data=train_dloader,
    val_data=val_dloader,
    epochs=100,
    log=True
)

_, acc = pipeline.single_validate(
    val_data=test_loader
)
np.array(acc).max()

Number of graphs: 1113
PROTEINS
Creating model...
Model created!
